<a href="https://colab.research.google.com/github/Kauboimausu/practica4-nlps/blob/main/practica4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from huggingface_hub import notebook_login
notebook_login()

# Práctica 4: Redes Neuronales Recurrentes y Procesamiento de Lenguaje Natural (NLP)

Esta práctica tiene como objetivo comprender el flujo de trabajo del NLP, desde la conversión de texto a representaciones numéricas (*tokens*), hasta el diseño, entrenamiento y evaluación crítica de una red neuronal recurrente (LSTM) aplicada al análisis de sentimiento en español.

**Conjunto de datos:** *IMDB Dataset of 50K Movie Reviews - Spanish*
(Descargar de: https://www.kaggle.com/datasets/luisdiegofv97/imdb-dataset-of-50k-movie-reviews-spanish)

**Instrucciones:**
> Implementa lo que se pide a continuación de manera individual. El código, las gráficas y las respuestas a las preguntas de análisis deben quedar plasmadas en este mismo Notebook. Se recomienda utilizar un subconjunto de los datos (ej. 10,000 reseñas).


In [9]:
import sys

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    %pip install -q torchmetrics

import torch
import torchvision
import numpy as np

In [10]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

---

## Parte 1: Del texto a tokens [1.5 puntos]

### Recordando qué es la tokenización
Los modelos de aprendizaje profundo no pueden procesar texto crudo (cadenas de caracteres). Asumen que el texto ha sido *tokenizado* y codificado como vectores numéricos. Históricamente, existen tres enfoques principales:

1. La _Character Tokenization_ (Nivel Carácter) se alimenta cada carácter individualmente al modelo. Ignora cualquier estructura lingüística. Su gran desventaja es que el modelo debe aprender a formar palabras desde cero, requiriendo demasiada memoria y datos.
2. La _Word Tokenization_ (Nivel Palabra) divide el texto usando espacios (ej. `text.split()`). El problema es que los signos de puntuación se adhieren a las palabras (ej. `"NLP."` se vuelve un token distinto a `"NLP"`). Dado que los idiomas tienen conjugaciones y declinaciones, el vocabulario crece a millones. Para evitar redes inmensas, se limita el vocabulario (ej. las 50,000 palabras más comunes) y el resto se clasifica como un token desconocido `[UNK]`. Esto causa pérdida de información.
3. La _Subword Tokenization_ (El estándar moderno) combina los mejores aspectos de los anteriores (usado por BERT, GPT, etc.). Divide las palabras raras en subunidades lingüísticas y mantiene enteras las palabras frecuentes. Por ejemplo, la palabra `"tokenizing"` podría dividirse en `"token"` y `"##izing"`.

### 1.1 Implementación y comparación [1.5 puntos]
Considera las siguientes tres oraciones complejas en español:

In [1]:
textos = [
    "dos horas de mi vida que no voy a recuperar!!! y ni siquiera aprendí nada...",
    "hmmm :( me gustó más pensar en la película que verla.",
    "que?? esto con 20 minutos menos sería otra cosa completamente distinta ajjaj."
]

**a)** Implementa una función básica de *Word Tokenization* en Python usando `.split()`. Muestra la lista de palabras generada para cada texto. Observa qué sucede con los signos de puntuación.

In [8]:
# [TU CÓDIGO AQUÍ] Word tokenization manual
palabras_conocidas = []
for review in textos:
    palabras_conocidas.extend(review.split())
diccionario = sorted(set(palabras_conocidas + ["[PAD]", "[UNK]"]))
word2idx = {palabra: idx for idx, palabra in enumerate(diccionario)}
idx2word = {idx: palabra for palabra, idx in word2idx.items()}
print(f"Diccionario conocido:")
for k, v in word2idx.items():
    print(f"{k!r:20s}  -->  {v}")

Diccionario conocido:
'20'                  -->  0
':('                  -->  1
'[PAD]'               -->  2
'[UNK]'               -->  3
'a'                   -->  4
'ajjaj.'              -->  5
'aprendí'             -->  6
'completamente'       -->  7
'con'                 -->  8
'cosa'                -->  9
'de'                  -->  10
'distinta'            -->  11
'dos'                 -->  12
'en'                  -->  13
'esto'                -->  14
'gustó'               -->  15
'hmmm'                -->  16
'horas'               -->  17
'la'                  -->  18
'me'                  -->  19
'menos'               -->  20
'mi'                  -->  21
'minutos'             -->  22
'más'                 -->  23
'nada...'             -->  24
'ni'                  -->  25
'no'                  -->  26
'otra'                -->  27
'película'            -->  28
'pensar'              -->  29
'que'                 -->  30
'que??'               -->  31
'recuperar!!!'        -->  3

El método __split__ toma como default splitter los espacios, entonces "nada..." es detectado cómo su propía palabra, diferente de "nada", "nada.", "nada..", etc... 

Esto claramente no es la forma óptima de tokenizar, al menos lógicamente tiene sentido para mí que los tokens sean "nada" y "..."

También es obvio ver cómo esto podría causar un problema para las risas, que pueden ser erraticas al escribirse por naturaleza de lo que indican.

**b)** usa la librería `transformers` de Hugging Face. Carga el tokenizador pre-entrenado en español `dccuchile/bert-base-spanish-wwm-uncased`. Aplícalo a las 3 oraciones e imprime los tokens resultantes (puedes usar el método `convert_ids_to_tokens`).
*   puedes leer sobre este tokenizador y el modelo asociado en su [página de Hugging Face](https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased).

In [12]:
#[TU CÓDIGO AQUÍ] Subword tokenization usando AutoTokenizer de transformers
from transformers import BertTokenizer

# importamos nuestro tokenizer preentrenado
tokenizador_espanol = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased", do_lower_case=False)

for review in textos:
    tokens = tokenizador_espanol.encode(review)
    print(f"{tokenizador_espanol.convert_ids_to_tokens(tokens)}")
    

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

['[CLS]', 'dos', 'horas', 'de', 'mi', 'vida', 'que', 'no', 'voy', 'a', 'recuperar', '!', '!', '!', 'y', 'ni', 'siquiera', 'aprendí', 'nada', '.', '.', '.', '[SEP]']
['[CLS]', 'hmm', '##m', ':', '(', 'me', 'gustó', 'más', 'pensar', 'en', 'la', 'película', 'que', 'verla', '.', '[SEP]']
['[CLS]', 'que', '?', '?', 'esto', 'con', '20', 'minutos', 'menos', 'sería', 'otra', 'cosa', 'completamente', 'distinta', 'aj', '##ja', '##j', '.', '[SEP]']





**c)** Responde, basado en los resultados impresos, ¿cómo manejó el tokenizador de HuggingFace los signos de puntuación, los acentos, etc? comenta todo lo que observas


## Parte 2: Preparación de datos y tensores [1.5 puntos]

Para poder procesar lotes (*batches*) de texto en una red neuronal, todas las secuencias deben tener la misma longitud. Para esto utilizamos **Padding** (relleno con ceros, representados por el token `[PAD]`) para las oraciones cortas, y **Truncamiento** para las oraciones demasiado largas.

**a)** Carga el archivo `.csv` de IMDB. Convierte las etiquetas de sentimiento a formato numérico (positivo = 1, negativo = 0).


**b)** Utiliza el `AutoTokenizer` del inciso anterior para tokenizar todo tu conjunto de datos. Asegúrate de aplicar `padding=True`, `truncation=True` y definir un `max_length` razonable (ej. 150 o 250 tokens).


**c)** Construye los objetos `TensorDataset` y `DataLoader` de PyTorch para tus conjuntos de entrenamiento (80%) y prueba (20%). Recuerda incluir tanto los `input_ids` como la etiqueta.


In [ ]:
# [TU CÓDIGO AQUÍ] Carga de pandas, tokenización masiva y creación de DataLoaders

## Parte 3: Construcción de la memoria (arquitectura LSTM) [2.5 puntos]

El mayor reto al programar Redes Neuronales Recurrentes es comprender cómo fluyen las dimensiones de los tensores a través del tiempo.

Implementa una clase `ModeloAnalisisSentimiento` que herede de `nn.Module`. Debe contener:
1. Una capa `nn.Embedding`.
2. Una capa `nn.LSTM` (puedes decidir si hacerla bidireccional, número de capas ocultas, etc.).
3. Una capa `nn.Linear` de salida.

Lo siguiente es obligatorio, tu método `forward` debe aceptar un parámetro booleano `print_shapes=False`. Cuando sea `True`, el modelo debe imprimir con un `print()` la dimensión del tensor (`.shape`) exactamente en estos tres momentos:
- A la salida de la capa de Embedding.
- A la salida de la capa LSTM (imprime la dimensión de `output`, del `hidden_state` y del `cell_state`).
- A la salida de la capa Lineal.

Una vez definida la clase, pasa un *batch* de prueba por tu modelo con `print_shapes=True` y **explica en texto** qué significa cada uno de los números en las dimensiones impresas (ej. *"El tamaño [32, 150, 128] significa[tamaño de batch, longitud de secuencia, dimensión del embedding]"*).

In [ ]:
import torch
import torch.nn as nn

class ModeloAnalisisSentimiento(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        # [TU CÓDIGO AQUÍ] Definir las capas: Embedding, LSTM, Dropout, Linear

    def forward(self, text, print_shapes=False):
        # Nota: La función de activación Sigmoid para obtener la probabilidad
        # se aplicará fuera del modelo, o implícitamente en la función de pérdida.
        # El modelo debe devolver los logits crudos.

        # [TU CÓDIGO AQUÍ] Implementar el forward pass y los prints condicionales
        pass

# [TU CÓDIGO AQUÍ] Instanciar el modelo y pasar un batch del dataloader con print_shapes=True


## Parte 4: Entrenamiento y evaluación [2 puntos]

Entrena la red LSTM.

**a)** Define tu optimizador (Adam recomendado) y tu función de pérdida (usualmente `BCEWithLogitsLoss` para clasificación binaria).


**b)** Realiza el ciclo de entrenamiento por un mínimo de 3 a 5 épocas.


**c)** Muestra en una gráfica cómo evoluciona la pérdida de entrenamiento y la de validación/prueba a lo largo de las épocas.


**d)** Reporta el *Accuracy* final en el conjunto de prueba.


In [ ]:
#[TU CÓDIGO AQUÍ] Bucle de entrenamiento y gráficas

## Parte 5: Análisis y "amnesia" temporal [1.5 puntos]


Las LSTM procesan la información de manera secuencial, asumiendo que el orden de las palabras importa. ¿Qué sucede si alteramos la naturaleza del tiempo en las oraciones?

**a)** Toma tu conjunto de prueba y **vueltea cronológicamente** el orden de todas las palabras en los textos (ej. *"La película fue mala"* $\to$ *"mala fue película La"*). Tokeniza este texto invertido y evalúalo usando tu modelo ya entrenado (NO vuelvas a entrenar el modelo).


**b)** Reporta el nuevo *Accuracy* sobre los datos invertidos.


**c)** **Análisis:** ¿El rendimiento cayó mucho al nivel del azar (50%) o se mantuvo relativamente alto? Explica teóricamente por qué la memoria oculta (*hidden state*) de tu LSTM reaccionó de esta manera.


In [ ]:
# [TU CÓDIGO AQUÍ] Invertir textos, generar DataLoader, evaluar y reportar Accuracy

## Parte 6: Hackeando tu propio modelo [1 puntos]

Un modelo puede tener métricas altísimas en el conjunto de prueba, pero fallar estrepitosamente en el lenguaje real del día a día debido al sarcasmo, ironía o frases coloquiales.

**a)** Inventa **5 reseñas originales (escritas por ti)**. El objetivo es que logres engañar a tu propio modelo.
- Si la reseña es positiva (para un humano), el modelo debe predecir equivocadamente "Negativo".
- Si la reseña es negativa (para un humano), el modelo debe predecir equivocadamente "Positivo".
- *Regla:* Tienen que ser frases coherentes en español, no palabras aleatorias. Intenta usar sarcasmo, falsos contrastes, modismos o frases complejas. (Ej. *"Una obra de arte perfecta si lo que buscas es dormir"*).

**b)** Pasa tus 5 frases por el modelo, imprime la probabilidad de predicción.


**c)** **Discusión:** Explica brevemente, para cada frase, por qué crees que el modelo falló. ¿Qué palabra o estructura confundió a las compuertas de actualización (update gates) de la LSTM y diluyó el verdadero sentimiento en el estado de la celda?


In [ ]:
#[TU CÓDIGO AQUÍ] Definir 5 textos originales adversarios, hacer inferencia e imprimir resultados